# 07 · One-sample sensorimotor constraints

Runs the do-next list from Project store `docs/sensorimotor-improvements-one-sample.md` on **one** emg2pose mini recording.

**Already used RNN?** Yes — notebook `05` / store `docs/05_lazzari_compositional_rnn_emg.ipynb` trains a Lazzari-style leaky RNN on all 30 mini files (~15.3° test MAE). This notebook asks what helps under **true n=1 file** scarcity.

Experiments:
1. Honest gap-split protocol + mean / ridge baselines
2. Kinematic soft losses on NMF→pose linear decode
3. Box 5.4 regularizers on a tiny leaky RNN
4. Synergy-space RNN vs raw-EMG RNN
5. Within-train augmentation
6. Leakage stress test (NMF on all time + shuffled windows)


In [ ]:
import os, sys, subprocess
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "setup.py").exists() and (cand / "emg2pose").exists():
            return cand
    raise FileNotFoundError("emg2pose repo root not found")

ROOT = find_repo_root(Path.cwd())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "emg2pose_dataset_mini"
if not DATA_DIR.exists():
    url = "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar"
    tar_path = ROOT / "emg2pose_dataset_mini.tar"
    print("Downloading mini dataset (~617 MB)...")
    subprocess.run(["curl", "-L", "--fail", "-o", str(tar_path), url], check=True)
    subprocess.run(["tar", "-xf", str(tar_path)], check=True)
    tar_path.unlink(missing_ok=True)

SCRIPT = ROOT / "notebooks/experiments/07_one_sample_constraints/run_one_sample_constraints.py"
print("repo:", ROOT)
print("hdf5 files:", len(list(DATA_DIR.glob('*.hdf5'))))
print("script:", SCRIPT)

In [ ]:
# Runs all ablations; writes data/one_sample_constraints/results.{json,md}
!python notebooks/experiments/07_one_sample_constraints/run_one_sample_constraints.py

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
import json

out = Path("data/one_sample_constraints")
display(Markdown((out / "results.md").read_text()))
summary = json.loads((out / "results.json").read_text())
rows = sorted(summary["results"], key=lambda r: r["test_mae_deg"])
print("\nBest → worst test MAE:")
for r in rows:
    print(f"  {r['test_mae_deg']:6.2f}°  {r['name']}")

## How to read this

- On **one recording**, beating mean pose by a lot is rare; treat ~0.5° moves cautiously.
- If `leaky_protocol_*` looks much better than honest gap-split ridge/linear, that gain is **leakage**, not a model win.
- Prefer NMF/ridge over RNN here unless synergy RNN clearly beats both mean and ridge on the honest split.
- The Lazzari `05` RNN used **all 30 mini files** — not comparable to this n=1 protocol.
